### Import & Data Load


In [1]:
# Import & Data Load
# (우리는 이미 로컬 가상환경에 필요한 패키지들을 설치 완료했습니다)

In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
train.head(5)

,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


### Check Data


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        3000 non-null   object 
 1   gender                    3000 non-null   object 
 2   age                       3000 non-null   int64  
 3   height                    3000 non-null   float64
 4   weight                    3000 non-null   float64
 5   cholesterol               3000 non-null   float64
 6   systolic_blood_pressure   3000 non-null   int64  
 7   diastolic_blood_pressure  3000 non-null   int64  
 8   glucose                   3000 non-null   float64
 9   bone_density              3000 non-null   float64
 10  activity                  3000 non-null   object 
 11  smoke_status              3000 non-null   object 
 12  medical_history           1711 non-null   object 
 13  family_medical_history    1514 non-null   object 
 14  sleep_pa

In [6]:
train.isnull().sum()

ID                             0
gender                         0
age                            0
height                         0
weight                         0
cholesterol                    0
systolic_blood_pressure        0
diastolic_blood_pressure       0
glucose                        0
bone_density                   0
activity                       0
smoke_status                   0
medical_history             1289
family_medical_history      1486
sleep_pattern                  0
edu_level                    607
mean_working                1032
stress_score                   0
dtype: int64

In [7]:
# 결측값 있는 칼럼(column) 확인
missing_columns_train = train.columns[train.isnull().sum() > 0]
missing_columns_train

Index(['medical_history', 'family_medical_history', 'edu_level',
       'mean_working'],
      dtype='object')

In [8]:
train[missing_columns_train].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   medical_history         1711 non-null   object 
 1   family_medical_history  1514 non-null   object 
 2   edu_level               2393 non-null   object 
 3   mean_working            1968 non-null   float64
dtypes: float64(1), object(3)
memory usage: 93.9+ KB


In [9]:
# 진단용 코드 셀 (필요 시 주피터 환경에서 수동 실행)

In [10]:
# 결측값이 있는 범주형 변수 탐색 (이전 결측값 수동 처리 완료)

# 범주형 변수의 결측값을 보다 정밀하게 대체합니다.
# 기존 병력(medical_history) 및 가족력(family_medical_history)의 결측은 특정 병력이 없음을 의미할 가능성이 큽니다.
train['medical_history'] = train['medical_history'].fillna('none')
test['medical_history'] = test['medical_history'].fillna('none')

train['family_medical_history'] = train['family_medical_history'].fillna('none')
test['family_medical_history'] = test['family_medical_history'].fillna('none')

# 학력의 결측값은 정보의 부재를 뜻하는 'Unknown' 클래스를 신설하여 대치합니다.
train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

In [11]:
# 1. 평균 근무 시간(mean_working) 결측치 처리
# 1-1. 결측 여부 자체를 피처로 추가 (is_working_missing)
train['is_working_missing'] = train['mean_working'].isnull().astype(int)
test['is_working_missing'] = test['mean_working'].isnull().astype(int)

# 1-2. 연령대별 중앙값으로 세분화 대치
train['age_group'] = (train['age'] // 10) * 10
test['age_group'] = (test['age'] // 10) * 10

age_working_medians = train.groupby('age_group')['mean_working'].median()
overall_median = train['mean_working'].median()

train['mean_working'] = train.apply(
    lambda row: age_working_medians.get(row['age_group'], overall_median) if pd.isnull(row['mean_working']) else row['mean_working'],
    axis=1
)
test['mean_working'] = test.apply(
    lambda row: age_working_medians.get(row['age_group'], overall_median) if pd.isnull(row['mean_working']) else row['mean_working'],
    axis=1
)

# 불필요해진 임시 연령대 컬럼 제거
train = train.drop('age_group', axis=1)
test = test.drop('age_group', axis=1)

# 2. 피처 엔지니어링 (핵심 수식 및 이상치 도메인 룰 반영)
# 체질량지수 (BMI) 생성
for df in [train, test]:
    df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# 맥압 (Pulse Pressure, PP) & 평균혈압 (Mean Arterial Pressure, MAP) 생성
for df in [train, test]:
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']
    df['map'] = (df['systolic_blood_pressure'] + 2 * df['diastolic_blood_pressure']) / 3

# 도메인 기반 고위험 위험군 플래그 생성
for df in [train, test]:
    df['is_extreme_overwork'] = (df['mean_working'] >= 12).astype(int)
    df['is_low_bone_density'] = (df['bone_density'] <= -1.0).astype(int)
    df['is_high_pulse_pressure'] = (df['pulse_pressure'] > 80).astype(int)

# 3. 명확한 계층/서열 구조가 있는 범주형 변수의 수동 인코딩 (Ordinal Encoding)
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
activity_map = {'light': 1, 'moderate': 2, 'intense': 3}

for df in [train, test]:
    df['edu_level_encoded'] = df['edu_level'].map(edu_map)
    df['activity_encoded'] = df['activity'].map(activity_map)

# 다중 신호 중복 방지를 위해 기존 원본 문자열 컬럼 드롭
train = train.drop(columns=['edu_level', 'activity'])
test = test.drop(columns=['edu_level', 'activity'])

In [12]:
# mean_working에 대해 중앙값 대체
median_value = train['mean_working'].median()
train['mean_working'] = train['mean_working'].fillna(median_value)
test['mean_working'] = test['mean_working'].fillna(median_value)

# --- Feature Engineering ---
# 1. BMI 지수 생성
for df in [train, test]:
    df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# 2. 맥압(Pulse Pressure) 및 평균혈압(Mean Blood Pressure) 생성
for df in [train, test]:
    df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']
    df['mean_pressure'] = df['diastolic_blood_pressure'] + df['pulse_pressure'] / 3

In [13]:
# 순서형 인코딩된 컬럼을 제외한 순수 범주형 데이터 변환 및 타입 지정
categorical_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']

for col in categorical_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [14]:
x_train = train.drop(['ID', 'stress_score'], axis = 1)
y_train = train['stress_score']

x_test = test.drop('ID', axis = 1)

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))
rmse_list = []

# LightGBMRegressor 학습 (그리드 서치로 튜닝된 최적 파라미터 적용)
for fold, (train_idx, val_idx) in enumerate(kf.split(x_train)):
    X_tr, y_tr = x_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = x_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # 복잡성을 제어하고 오버피팅을 억제하는 lr=0.05, leaves=15 세팅
    model = LGBMRegressor(
        learning_rate=0.05, 
        num_leaves=15, 
        min_child_samples=20,
        n_estimators=300, 
        random_state=42, 
        verbose=-1
    )
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    rmse = root_mean_squared_error(y_val, val_preds)
    rmse_list.append(rmse)
    
    # Fold별 테스트 예측값 누적 평균
    test_preds += model.predict(x_test) / 5
    print(f'Fold {fold+1} RMSE: {rmse:.6f}')

print(f'==> 5-Fold Average RMSE: {np.mean(rmse_list):.6f}')

In [15]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))
mae_list = []

# LightGBMRegressor 학습 (MAE 평가지표를 위한 regression_l1 최적 파라미터 적용)
for fold, (train_idx, val_idx) in enumerate(kf.split(x_train)):
    X_tr, y_tr = x_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = x_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # MAE 오차를 다이렉트로 학습 최소화하는 L1 regression 설정
    model = LGBMRegressor(
        objective='regression_l1',
        learning_rate=0.05, 
        num_leaves=15, 
        min_child_samples=20,
        n_estimators=300, 
        random_state=42, 
        verbose=-1
    )
    model.fit(X_tr, y_tr)
    
    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    mae = mean_absolute_error(y_val, val_preds)
    mae_list.append(mae)
    
    # Fold별 테스트 예측값 누적 평균
    test_preds += model.predict(x_test) / 5
    print(f'Fold {fold+1} MAE: {mae:.6f}')

print(f'==> 5-Fold Average MAE: {np.mean(mae_list):.6f}')

Fold 1 MAE: 0.213557


Fold 2 MAE: 0.217507


Fold 3 MAE: 0.226429


Fold 4 MAE: 0.231494


Fold 5 MAE: 0.223558
==> 5-Fold Average MAE: 0.222509


### Submission


In [16]:
submission = pd.read_csv('sample_submission.csv')

In [17]:
submission['stress_score'] = test_preds
submission.head()

,ID,stress_score
0,TEST_0000,0.482331
1,TEST_0001,0.524561
2,TEST_0002,0.277534
3,TEST_0003,0.457471
4,TEST_0004,0.587249


In [18]:
submission.to_csv('submit.csv', index=False)